# Surrogate Training GPU Smoke Test

This notebook runs the small supervised surrogate-training smoke benchmark on a Colab GPU. It does not install or run Julia on Colab. Compare the printed JSON against a local Julia CPU run of the same benchmark command.

Before running, select `Runtime -> Change runtime type -> GPU`. The setup cell asserts that JAX sees a GPU and fails loudly if Colab is on CPU.

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_DSGE.git"
BRANCH = "codex/nonlinear-sep-surrogate-port"
ROOT = Path("/content/SurrogateNN_DSGE")
JAX_BACKEND = "auto"  # "auto", "cuda12", "cuda13", or "cpu".
STRICT_GPU = True
FORCE_REINSTALL_JAX = True


def run(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc


if "jax" in sys.modules:
    raise RuntimeError("JAX is already imported. Restart the Colab runtime before reinstalling CUDA JAX wheels.")

nvidia_smi = run(["bash", "-lc", "nvidia-smi || true"], check=False).stdout
cuda_match = re.search(r"CUDA Version:\\s*([0-9]+)", nvidia_smi)
cuda_major = int(cuda_match.group(1)) if cuda_match else None
print("Detected CUDA major:", cuda_major)

if STRICT_GPU and cuda_major is None:
    raise RuntimeError("nvidia-smi did not report CUDA. Use a Colab GPU runtime before continuing.")

os.environ["JAX_ENABLE_X64"] = "1"
if JAX_BACKEND == "cpu":
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
else:
    os.environ.pop("JAX_PLATFORMS", None)
    os.environ.pop("JAX_PLATFORM_NAME", None)
    removed_ld_library_path = os.environ.pop("LD_LIBRARY_PATH", None)
    if removed_ld_library_path:
        print("Cleared LD_LIBRARY_PATH so pip-installed CUDA libraries take precedence.")

if ROOT.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=ROOT)
    run(["git", "checkout", BRANCH], cwd=ROOT)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=ROOT)
else:
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)])

os.chdir(ROOT)
print("Working directory:", Path.cwd())

if JAX_BACKEND == "auto":
    selected_jax_extra = "cuda13" if (cuda_major is not None and cuda_major >= 13) else "cuda12"
else:
    selected_jax_extra = JAX_BACKEND
if selected_jax_extra not in {"cpu", "cuda12", "cuda13"}:
    raise ValueError(f"Unsupported JAX_BACKEND={selected_jax_extra!r}")
print("Selected JAX install target:", selected_jax_extra)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
if FORCE_REINSTALL_JAX:
    run([
        sys.executable, "-m", "pip", "uninstall", "-y",
        "jax", "jaxlib", "jax-cuda12-pjrt", "jax-cuda12-plugin",
        "jax-cuda13-pjrt", "jax-cuda13-plugin",
    ], check=False)

jax_package = "jax" if selected_jax_extra == "cpu" else f"jax[{selected_jax_extra}]"
run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", jax_package])
run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "sympy>=1.13,<2", "scipy>=1.10,<2", "numpyro>=0.20", "pytest>=8.0,<9",
])
run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "show", "jax", "jaxlib", "numpyro"], check=False)

sys.path.insert(0, str(ROOT / "src"))

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import numpyro

print("Python:", sys.version)
print("JAX:", jax.__version__)
print("NumPyro:", numpyro.__version__)
print("JAX x64 enabled:", jax.config.read("jax_enable_x64"))
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())

gpu_devices = [device for device in jax.devices() if device.platform in {"gpu", "cuda"}]
if selected_jax_extra != "cpu" and not gpu_devices:
    raise RuntimeError("CUDA JAX was requested, but JAX did not report a GPU device.")
if selected_jax_extra != "cpu":
    probe = (jnp.ones((256, 256), dtype=jnp.float32) @ jnp.ones((256, 256), dtype=jnp.float32)).block_until_ready()
    print("GPU probe result:", float(probe[0, 0]))


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

output_path = Path("benchmarks/results/colab_gpu_surrogate_training_smoke.json")
cmd = [
    sys.executable,
    "benchmarks/surrogate_training_smoke.py",
    "--mode", "python",
    "--architecture", "both",
    "--device", "gpu",
    "--theta-draws", "4",
    "--periods", "12",
    "--epochs", "8",
    "--hidden", "8",
    "--batch-size", "16",
    "--output", str(output_path),
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, check=False, text=True, capture_output=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f"surrogate training smoke benchmark failed with return code {proc.returncode}")

result = json.loads(output_path.read_text())
print(json.dumps(result, indent=2, sort_keys=True))
for architecture, section in result["python"].items():
    assert section["status"] == "ok", (architecture, section)
    assert section["jax_backend"] in {"gpu", "cuda"}, section
    assert section["bundle_roundtrip_max_abs"] <= 1e-10, section
print("GPU surrogate training smoke test passed.")


Local Julia CPU comparison command:

```bash
.venv/bin/python benchmarks/surrogate_training_smoke.py \
  --mode both --architecture both --device cpu \
  --theta-draws 4 --periods 12 --epochs 8 --hidden 8 --batch-size 16
```
